# PennyLane QAOA for MaxCut

Evaluate a p=1 triangle-graph QAOA landscape with the same Hamiltonian on both devices.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

QAOA represents a combinatorial objective as a cost Hamiltonian and alternates cost and mixer evolution.

In [2]:
cost = 1.5 * qml.I(0) - 0.5 * (qml.Z(0) @ qml.Z(1)) - 0.5 * (qml.Z(1) @ qml.Z(2)) - 0.5 * (qml.Z(0) @ qml.Z(2))
parameters = [(gamma, beta) for gamma in np.linspace(0.0, np.pi, 7) for beta in np.linspace(0.0, np.pi / 2, 5)]

def make_qnode(device):
    @qml.qnode(device)
    def circuit(gamma, beta):
        for wire in range(3):
            qml.Hadamard(wire)
        for wires in ((0, 1), (1, 2), (0, 2)):
            qml.IsingZZ(-gamma, wires=wires)
        for wire in range(3):
            qml.RX(2 * beta, wires=wire)
        return qml.expval(cost)
    return circuit

reference_qnode = make_qnode(qml.device("default.qubit", wires=3))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(*values) for values in parameters]))

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(*values) for values in parameters]))
error = max_abs_error(reference, candidate)
best_match = int(np.argmax(reference)) == int(np.argmax(candidate))
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

The complete parameter landscape must agree between devices.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/06_qaoa_maxcut.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="QAOA landscape atol=4e-6",
    passed=error <= 4e-6 and best_match,
    exact_match=best_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_cost_error": error, "best_parameters": parameters[int(np.argmax(candidate))], "best_cost": candidate.max()},
)


Comparison summary
------------------
Correctness contract: PASS — QAOA landscape atol=4e-6
SDK reference median: 31.589 ms
MettleQ median:       105.114 ms
Timing interpretation: the SDK reference was 3.328x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "QAOA landscape atol=4e-6", "exact_match": true, "framework": "pennylane", "machine": "arm64", "metrics": {"best_cost": 1.9620181322097778, "best_parameters": [0.5235987755982988, 0.39269908169872414], "max_cost_error": 1.0100355170017394e-06}, "mettleq_median_ms": 105.1137079775799, "notebook": "pennylane/06_qaoa_maxcut.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 31.58904198789969, "reference_over_mettleq": 0.30052257308473446, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Larger graphs may justify acceleration, while this three-wire landscape remains overhead-bound.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.